# Train v22 on Workstation

Run only the cells in this notebook. They do not start profiling, preprocessing, or training.

The notebook extracts the package to a local runtime folder, lets you confirm paths and parameters, then generates PowerShell scripts. Run those scripts from a terminal so progress, failures, and logs are fully visible.

Generated scripts: `run_install_deps.ps1`, `run_profile.ps1`, `run_preprocess.ps1`, and `run_train.ps1`.

In [ ]:
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

VERSION = 'v22'
DEFAULT_DRIVE_CODE_DIR = Path('H:/My Drive/quant-research-workbench/workstation_code/v22')
DRIVE_CODE_DIR = Path(os.environ.get('QW_DRIVE_CODE_DIR', str(DEFAULT_DRIVE_CODE_DIR)))
PACKAGE_ZIP = DRIVE_CODE_DIR / f'inhouse_transformer_{VERSION}_workstation.zip'
MANIFEST_PATH = DRIVE_CODE_DIR / 'workstation_manifest.json'
LOCAL_CODE_ROOT = Path(f'D:/TradingCodes/quant-research-workbench-{VERSION}-runtime')

assert PACKAGE_ZIP.exists(), f'Missing package: {PACKAGE_ZIP}'
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text())
print(json.dumps(manifest, indent=2))

if LOCAL_CODE_ROOT.exists():
    shutil.rmtree(LOCAL_CODE_ROOT)
LOCAL_CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as package:
    package.extractall(LOCAL_CODE_ROOT)
sys.path.insert(0, str(LOCAL_CODE_ROOT))
print('installed code at', LOCAL_CODE_ROOT)


Confirm these paths before generating commands. Edit this cell if your flatfiles, cache, or model output should live somewhere else on the workstation.

In [ ]:
# Edit FLATFILES_ROOT if you copy data from the HDD/Drive path to local SSD/NVMe.
FLATFILES_ROOT = Path(manifest['default_flatfiles_root'])
CANONICAL_ROOT = Path(manifest.get('default_canonical_root') or (FLATFILES_ROOT / 'derived' / 'canonical_events_v1'))
CACHE_ROOT = Path(manifest['default_cache_root'])
OUTPUT_ROOT = Path(manifest['default_output_root'])
print('flatfiles root:', FLATFILES_ROOT, 'exists=', FLATFILES_ROOT.exists())
print('canonical root:', CANONICAL_ROOT)
print('cache root:', CACHE_ROOT)
print('output root:', OUTPUT_ROOT)
CANONICAL_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


Generate the terminal scripts. This cell creates all scripts in the extracted runtime folder and prints the exact PowerShell commands to run. It does not start any long-running job.

In [ ]:
# Configure these values, then run this cell once to create all PowerShell scripts.
PROFILE_CHUNK_MS = '100,250,500,1000'
PROFILE_CAPS = '64,128,256,512'
CHUNK_MS = '500'
MAX_QUOTE_EVENTS = 128
MAX_TRADE_EVENTS = 192
MAX_TOTAL_EVENTS = 256
PROFILE_SESSIONS = int(manifest.get('default_profile_sessions', 4))
PROFILE_PROCESSES = int(manifest.get('default_profile_processes', 2))
PREPROCESS_PROCESSES = int(manifest.get('default_preprocess_processes', 8))
POLARS_THREADS_PER_PROCESS = int(manifest.get('default_polars_threads_per_process', 2))
REBUILD_PREPROCESS_CACHE = False
BATCH_SIZE = int(manifest.get('default_batch_size', 4096))
EPOCHS = int(manifest.get('default_epochs', 3))
NUM_WORKERS = int(manifest.get('default_num_workers', 8))
PREFETCH_FACTOR = int(manifest.get('default_prefetch_factor', 4))
MAX_STEPS = 0
COUNT_COVERAGE = False
DRY_RUN = False
REBUILD_CACHE = False

RUN_LOG_DIR = OUTPUT_ROOT / 'workstation_logs'
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)

def ps_quote(value):
    text = str(value).replace('`', '``').replace('"', '`"')
    return f'"{text}"'

def terminal_command(script_path, args):
    return ' '.join(['&', ps_quote(sys.executable), '-u', ps_quote(script_path), *[ps_quote(arg) for arg in args]])

def write_command_script(label, script_path, args):
    script_path = Path(script_path)
    if not script_path.exists():
        raise FileNotFoundError(
            f'Missing {label} script: {script_path}. Run the extraction cell above before generating commands.'
        )
    command = terminal_command(script_path, args)
    ps1_path = LOCAL_CODE_ROOT / f'run_{label}.ps1'
    log_path = RUN_LOG_DIR / f'{VERSION}_{label}.log'
    ps1_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    py_path = str(LOCAL_CODE_ROOT).replace("'", "''")
    ps1 = (
        "$ErrorActionPreference = 'Stop'\n"
        "$env:PYTHONUNBUFFERED = '1'\n"
        f"$env:PYTHONPATH = '{py_path}' + [System.IO.Path]::PathSeparator + $env:PYTHONPATH\n"
        f"{command} 2>&1 | Tee-Object -FilePath {ps_quote(log_path)}\n"
        "if ($LASTEXITCODE -ne 0) { throw \"Command failed with exit code $LASTEXITCODE\" }\n"
    )
    ps1_path.write_text(ps1, encoding='utf-8')
    return ps1_path, log_path, command

def print_script(label, ps1_path, log_path, command):
    print('=' * 96)
    print(f'{label.upper()} SCRIPT')
    print('PowerShell script:', ps1_path)
    print('Log file:', log_path)
    print('Run this in PowerShell:')
    print('& ' + ps_quote(ps1_path))
    print('Direct command equivalent:')
    print(command)
    print('=' * 96)

def add_rebuild_flag(args, enabled):
    if enabled:
        args.append('--rebuild-cache')

PROFILE_ENABLED = True

install_ps1 = LOCAL_CODE_ROOT / 'run_install_deps.ps1'
install_log = RUN_LOG_DIR / f'{VERSION}_install_deps.log'
install_command = f'& {ps_quote(sys.executable)} -m pip install polars pyarrow wandb torchinfo torchview graphviz'
install_ps1.parent.mkdir(parents=True, exist_ok=True)
install_log.parent.mkdir(parents=True, exist_ok=True)
install_ps1.write_text(
    "$ErrorActionPreference = 'Stop'\n"
    f"{install_command} 2>&1 | Tee-Object -FilePath {ps_quote(install_log)}\n"
    "if ($LASTEXITCODE -ne 0) { throw \"Command failed with exit code $LASTEXITCODE\" }\n",
    encoding='utf-8',
)
print_script('install_deps', install_ps1, install_log, install_command)

profile_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'profile_event_chunks.py'
profile_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['validation_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', PROFILE_CHUNK_MS,
    '--caps', PROFILE_CAPS,
    '--max-profile-sessions', str(PROFILE_SESSIONS),
    '--processes', str(PROFILE_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]

preprocess_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'preprocess_event_chunks.py'
preprocess_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--canonical-root', str(CANONICAL_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['test_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', CHUNK_MS,
    '--max-quote-events', str(MAX_QUOTE_EVENTS),
    '--max-trade-events', str(MAX_TRADE_EVENTS),
    '--max-total-events', str(MAX_TOTAL_EVENTS),
    '--processes', str(PREPROCESS_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]
add_rebuild_flag(preprocess_args, REBUILD_PREPROCESS_CACHE)

train_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'train.py'
args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--canonical-root', str(CANONICAL_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--train-start-date', manifest['train_start_date'],
    '--train-end-date', manifest['train_end_date'],
    '--validation-start-date', manifest['validation_start_date'],
    '--validation-end-date', manifest['validation_end_date'],
    '--test-start-date', manifest['test_start_date'],
    '--test-end-date', manifest['test_end_date'],
    '--device', 'cuda',
    '--output-root', str(OUTPUT_ROOT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--tickers', manifest.get('tickers', 'ALL'),
    '--checkpoint-policy', 'last_only',
    '--wandb-entity', manifest['wandb_entity'],
    '--wandb-project', manifest['wandb_project'],
    '--wandb-run-name', manifest['wandb_run_name'],
    '--output-name', manifest['wandb_run_name'],
]
if REBUILD_CACHE:
    args.append('--rebuild-cache')
if COUNT_COVERAGE:
    args.append('--count-coverage')
if DRY_RUN:
    args.append('--dry-run')

generated = {}
if PROFILE_ENABLED:
    generated['profile'] = write_command_script('profile', profile_py, profile_args)
generated['preprocess'] = write_command_script('preprocess', preprocess_py, preprocess_args)
generated['train'] = write_command_script('train', train_py, args)
commands_md = LOCAL_CODE_ROOT / 'WORKSTATION_COMMANDS.md'
lines = [
    f'# {VERSION} workstation commands',
    '',
    'Run these from PowerShell in order:',
    '',
    f'1. `& {ps_quote(install_ps1)}` if dependencies are missing.',
]
for index, key in enumerate(['profile', 'preprocess', 'train'], start=2):
    if key in generated:
        lines.append(f'{index}. `& {ps_quote(generated[key][0])}`')
lines.extend(['', f'Logs: `{RUN_LOG_DIR}`', ''])
commands_md.write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))
print('\nCommand summary written to:', commands_md)
for label, (ps1_path, log_path, command) in generated.items():
    print_script(label, ps1_path, log_path, command)


Run the generated scripts from PowerShell in this order:

1. `run_install_deps.ps1` if dependencies are missing in the Python environment used by the scripts.
2. `run_profile.ps1`
3. `run_preprocess.ps1`
4. `run_train.ps1`

Each script writes a log under `OUTPUT_ROOT / 'workstation_logs'`.